# G32 — Diagnostics after G31: oracle mirror and an A-vs-B mixture (linear, train+val, test untouched)

**Why.** G31 (step 1) did not pass, but it was uninformative about the mechanism: its gate collapsed to ≈ 0 in every
fold, the regularisation sat at the grid edge, p_mirror was flattened, and its "rest" expert contained A's own clip-III
turn. **This is a second look after seeing G31**, and it is reported as such.

**Shared fixes.** L2 grid for the expert weights W {1e-4, 1e-3, 1e-2, 1e-1, 1}; a **separate** L2 grid for the gate
weights v {0, 1e-4, 1e-3, 1e-2}; p_mirror's C grid {0.001, 0.01, 0.1, 1, 10}; the gate starts at the training-fold
mirror share (logit of P(Y_B = Y_A) ≈ 0.355) instead of 0.5. All choices by grouped inner CV on NLL.

**Mixture (both diagnostics):** p = g·p_mirror + (1 − g)·p_own, with
* p_own = softmax(W[x_L3, x_L12] + b) — **B's (L's) evidence only**;
* g = σ(vᵀ[x_A, x_L3, x_L12, x_D2, x_D1, text III, audio III, log p_mirror] + c).

**Additive control (same raw information):** softmax(α·log p_mirror + W[x_A, x_L3, x_L12, x_D2, x_D1, text III, audio III] + b).

| Diagnostic | p_mirror |
|---|---|
| **(i) oracle mirror** (upper bound, analysis only) | (1 − ε)·one-hot(A's gold clip-III label) + ε/7; ε ∈ {0.1, 0.3, 0.5, 0.7, 0.9} chosen by each model's own inner CV (jointly with its L2 strengths) |
| **(ii) A-vs-B split** | logistic P(Y_A ∣ x_A, text III, audio III) trained on A's label of the training rows (cross-fitted on training rows) — **A's evidence only** |

**Reading rule (fixed before running), for each diagnostic:** passed iff (a) ΔNLL = NLL(additive) − NLL(mixture) has a
CI entirely above 0 **and** (b) the AUC of the out-of-fold gate for Y_B = Y_A has a CI lower bound above 0.5.

| (i) oracle | (ii) A-vs-B | Conclusion |
|---|---|---|
| not passed | any | even with A's true emotion the gate cannot tell when B mirrors → **stop direction A** |
| passed | not passed | the structure has value, the bottleneck is recognising A's emotion → contribution 2 must improve the A branch |
| passed | passed | write the 2×2 notebook (role tokens vs noRole × one query vs mechanism queries) |

Also reported: gate distribution (does it still collapse?), the chosen grid values and whether they hit a grid edge.
If a chosen value sits on an edge, the conclusion is stated as limited by that.

In [ ]:
# ======== CONFIG ========
import os


def first_existing(*paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"none of {paths}")


DATASET_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
FEATURES_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2"
SPLIT_CSV = first_existing("/kaggle/input/datasets/ptrnghieu/hi-ef-split/source_folder_split_seed42.csv",
                           "/kaggle/input/hi-ef-split/source_folder_split_seed42.csv")
G8A_DIR = first_existing("/kaggle/input/datasets/ptrnghieu/g8a-features", "/kaggle/input/g8a-features")
OUT_DIR = "/kaggle/working"

N_OUTER, N_INNER_DEV = 5, 5
SEEDS = [42, 123, 456]                       # as G8b / G11 / G12
LR, WEIGHT_DECAY = 1e-4, 1e-5                # B1 settings (only needed by the shared model cell)
FC_EPOCHS, PATIENCE, FC_BATCH = 50, 8, 32
RN = dict(D=128, heads=4, layers=2, dropout=0.2, lr=3e-4, wd=1e-2, epochs=80, patience=12, batch=64,
          aux_w=0.3, a_w=0.3, p_drop_ctx=0.3, p_drop_face=0.15)     # G8b, unchanged
PCA_DIM, MAXF, MAXF_POOL = 128, 24, 32
SAME_PERSON_COS, DOMINANT_MIN_FRAC = 0.45, 0.25
VOICE_SAME_COS = 0.35
DEBUG_PER_EPISODE = None     # e.g. 6 for a quick smoke test

FULL = dict(role=True, faces=True, ctx=True, aux=True, mdrop=True)
BASE = dict(readout='query', nq=1, q_readonly=False, role_emb=True, clip_emb=True, absent='token',
            drop=(), speech_parts=('text', 'audio', 'voice'), clips=(0, 1, 2), seed_offset=0)
EXPERIMENTS = [("RoleNet", 'role', FULL)]    # only used by the shared model cell's parameter print
N_BOOT = 2000
PCA_TXT = PCA_AUD = 32
LAM_W = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]
LAM_V = [0.0, 1e-4, 1e-3, 1e-2]
C_GRID = [0.001, 0.01, 0.1, 1.0, 10.0]
EPS_GRID = [0.1, 0.3, 0.5, 0.7, 0.9]

In [ ]:
import os, json, math, random, time
import numpy as np
import pandas as pd

EMO = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
POL = ['positive', 'neutral', 'negative']
E2I = {e: i for i, e in enumerate(EMO)}
P2I = {p: i for i, p in enumerate(POL)}

# Reference numbers from the locked-split report (validation, 5-seed mean)
REPORT_REF = {'B1_full': (24.65, 35.79), 'T1_future_KL': (25.34, 35.65),
              'Frozen A recognizer (E_A)': (23.21, 33.41)}


def load_tables(annot_csv, split_csv):
    """annotation.csv has no header: 0 clip_id, 1 text, 5 polarity, 6 intensity, 7 emotion, 8 uncertainty."""
    ann = pd.read_csv(annot_csv, header=None, dtype=str).set_index(0)
    sp = pd.read_csv(split_csv, dtype=str)

    def text(c):
        t = ann.at[c, 1] if c in ann.index else None
        return t if isinstance(t, str) else ''

    for k in (1, 2, 3):
        sp[f't{k}'] = sp[f'clip{k}'].map(text)
    sp['yA'] = sp['clip3_emotion'].map(E2I)
    sp['yB'] = sp['clip4_emotion'].map(E2I)
    sp['pA'] = sp['clip3'].map(lambda c: P2I.get(ann.at[c, 5], -1))
    assert sp[['yA', 'yB']].notna().all().all(), 'missing A/B emotion labels'
    return ann, sp


def eval_rows(sp, split, unlock_test=False):
    if split == 'test' and not unlock_test:
        raise RuntimeError('Test split is locked. Set UNLOCK_TEST = True only for the final, preregistered run.')
    return sp[sp['split'] == split].reset_index(drop=True)


def war_uar(pred, y, k):
    pred, y = np.asarray(pred), np.asarray(y)
    war = (pred == y).mean() * 100
    uar = np.mean([(pred[y == c] == c).mean() * 100 for c in range(k) if (y == c).any()])
    return war, uar


def source_boot_ci(pred, y, src, k, n_boot=2000, seed=0):
    """95% CI by resampling whole source folders (episodes) with replacement."""
    pred, y, src = np.asarray(pred), np.asarray(y), np.asarray(src)
    rng = np.random.default_rng(seed)
    groups = [np.where(src == s)[0] for s in np.unique(src)]
    stats = []
    for _ in range(n_boot):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        stats.append(war_uar(pred[idx], y[idx], k))
    lo, hi = np.percentile(np.array(stats), [2.5, 97.5], axis=0)
    return lo, hi


def report(name, pred, y, src, k=7):
    war, uar = war_uar(pred, y, k)
    lo, hi = source_boot_ci(pred, y, src, k)
    print(f'{name:<46} UAR {uar:5.2f} [{lo[1]:5.1f},{hi[1]:5.1f}]   WAR {war:5.2f} [{lo[0]:5.1f},{hi[0]:5.1f}]')
    return {'name': name, 'UAR': uar, 'WAR': war, 'UAR_lo': lo[1], 'UAR_hi': hi[1], 'WAR_lo': lo[0], 'WAR_hi': hi[0]}


def transition_tables(train_rows, alpha=1.0):
    """P(B | E_A) and P(B | E_A, P_A) estimated on TRAIN gold pairs, add-alpha smoothing."""
    T = np.full((7, 7), alpha)
    TP = np.full((7, 3, 7), alpha)
    for a, p, b in zip(train_rows['yA'], train_rows['pA'], train_rows['yB']):
        T[a, b] += 1
        if p >= 0:
            TP[a, p, b] += 1
    return T / T.sum(1, keepdims=True), TP / TP.sum(2, keepdims=True)


def rtt_forecast(pA_emo, T, pA_pol=None, TP=None):
    """Recognize-then-Transition: B distribution from A posteriors.
    Returns hard (argmax of transition row of argmax A) and soft (expected) B predictions."""
    hard = T[pA_emo.argmax(1)].argmax(1)
    if pA_pol is not None and TP is not None:
        pB = np.einsum('na,np,apb->nb', pA_emo, pA_pol, TP)  # assumes E_A and P_A posteriors independent
    else:
        pB = pA_emo @ T
    return hard, pB.argmax(1), pB

import glob, pickle
import torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ANNOT_CSV = glob.glob(os.path.join(DATASET_DIR, "*", "Hi-EF", "annotation.csv"))[0]
ann, sp = load_tables(ANNOT_CSV, SPLIT_CSV)
sp = sp[sp.split.isin(['train', 'val'])].reset_index(drop=True)      # test rows are dropped here
assert 'test' not in set(sp.split)
if DEBUG_PER_EPISODE:
    sp = sp.groupby('source_folder', group_keys=False).head(DEBUG_PER_EPISODE).reset_index(drop=True)
DEV = sp
train_all = ev = DEV          # names used by the shared feature-loading cell
N = len(DEV)
EPS = np.array(sorted(DEV.source_folder.unique()))
print(f"development MCIS {N} | episodes {len(EPS)}")


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
# ---- load every clip used by any MCIS (I-IV) once, keep it on the GPU
all_clips = sorted(set(sp[['clip1', 'clip2', 'clip3', 'clip4']].values.ravel()) & set(
    f[:-3].replace('_', '/', 1) for f in os.listdir(FEATURES_DIR) if f.endswith('.pt')))
CIDX = {c: i for i, c in enumerate(all_clips)}
missing = [c for c in set(train_all[['clip1', 'clip2', 'clip3']].values.ravel()) | set(ev[['clip1', 'clip2', 'clip3']].values.ravel())
           if c not in CIDX]
assert not missing, f"{len(missing)} clips without features, e.g. {missing[:3]}"

bufs = {k: [] for k in ('face', 'fmask', 'ori', 'text', 'audio', 'afound')}
for c in tqdm(all_clips, desc='loading features'):
    d = torch.load(os.path.join(FEATURES_DIR, c.replace('/', '_') + '.pt'), map_location='cpu', weights_only=False)
    face = d['face_features'].float()
    fm = d.get('face_valid_mask')
    bufs['face'].append(face)
    bufs['fmask'].append(torch.ones(face.shape[0], dtype=torch.bool) if fm is None else torch.as_tensor(fm).bool().reshape(-1))
    bufs['ori'].append(d['ori_features'].float())
    bufs['text'].append(d['text_feature'].float().reshape(-1))
    bufs['audio'].append(d.get('audio_feature', torch.zeros(527)).float().reshape(-1))
    bufs['afound'].append(torch.tensor(bool(d.get('audio_found', True))))
FEAT = {k: torch.stack(v).to(DEVICE) for k, v in bufs.items()}
del bufs
print({k: tuple(v.shape) for k, v in FEAT.items()})


def gather(idx):
    """idx: LongTensor of clip indices (any shape) -> dict of feature tensors with that leading shape."""
    flat = idx.reshape(-1)
    return {k: v[flat].reshape(*idx.shape, *v.shape[1:]) for k, v in FEAT.items()}

In [ ]:
class TemporalEncoder(nn.Module):
    def __init__(self, d=512, n_frames=16, layers=2, heads=8, dropout=0.1):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1, n_frames, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)

    def forward(self, x, mask):  # mask: True = valid frame
        mask = mask.clone()
        mask[~mask.any(1), 0] = True
        h = self.enc(x + self.pos[:, :x.size(1)], src_key_padding_mask=~mask)
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1)


class ClipEncoder(nn.Module):
    """Face/original temporal encoders + text/audio tokens -> 1-layer fusion Transformer -> one 512-d vector."""

    def __init__(self, d=512):
        super().__init__()
        self.face, self.ori = TemporalEncoder(d), TemporalEncoder(d)
        self.text = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, d))
        self.audio = nn.Sequential(nn.LayerNorm(527), nn.Linear(527, d))
        self.modality = nn.Parameter(torch.randn(1, 4, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 8, 4 * d, 0.1, batch_first=True, norm_first=True)
        self.fusion = nn.TransformerEncoder(layer, 1, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)

    def forward(self, b):
        ori_mask = torch.ones(b['ori'].shape[:2], dtype=torch.bool, device=b['ori'].device)
        tokens = torch.stack([self.face(b['face'], b['fmask']), self.ori(b['ori'], ori_mask),
                              self.text(b['text']), self.audio(F.normalize(b['audio'], dim=-1))], 1)
        valid = torch.ones(tokens.shape[:2], dtype=torch.bool, device=tokens.device)
        valid[:, 3] = b['afound']
        h = self.fusion(tokens + self.modality, src_key_padding_mask=~valid)
        m = valid.unsqueeze(-1).float()
        return self.norm((h * m).sum(1) / m.sum(1))


class ClipRecognizer(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.enc = ClipEncoder(d)
        self.drop = nn.Dropout(0.3)
        self.emo, self.pol = nn.Linear(d, 7), nn.Linear(d, 3)

    def forward(self, b):
        h = self.drop(self.enc(b))
        return self.emo(h), self.pol(h)


N_REC = 12   # 7 emotion probs + 3 polarity probs + max prob + entropy


class Forecaster(nn.Module):
    def __init__(self, use_raw=True, use_traj=False, d=512, positions=None):
        super().__init__()
        self.use_raw, self.use_traj = use_raw, use_traj
        self.positions = positions   # clip positions (0=I, 1=II, 2=III); None = the last n clips
        self.enc = ClipEncoder(d) if use_raw else None
        self.traj = nn.Sequential(nn.LayerNorm(N_REC), nn.Linear(N_REC, d), nn.GELU(), nn.Linear(d, d)) if use_traj else None
        self.clip_pos = nn.Parameter(torch.randn(1, 3, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 8, 4 * d, 0.1, batch_first=True, norm_first=True)
        self.inter = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, d // 2), nn.GELU(),
                                  nn.Dropout(0.2), nn.Linear(d // 2, 7))

    def forward(self, clip_idx, rec):  # clip_idx [B,n], rec [B,n,N_REC], n <= 3 clips in temporal order
        B, n = clip_idx.shape
        tok = 0
        if self.use_raw:
            feats = gather(clip_idx)
            flat = {k: v.reshape(B * n, *v.shape[2:]) for k, v in feats.items()}
            tok = self.enc(flat).reshape(B, n, -1)
        if self.use_traj:
            tok = tok + self.traj(rec)
        pos = self.clip_pos[:, list(self.positions)] if self.positions is not None else self.clip_pos[:, 3 - n:]
        h = self.inter(tok + pos)
        return self.head(h.mean(1))

## From G8a features to role-tagged slots

In [ ]:
from collections import defaultdict
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA

G8 = {}
for f in sorted(glob.glob(os.path.join(G8A_DIR, '**', 'shard_*.pkl'), recursive=True)):
    G8.update(pickle.load(open(f, 'rb')))
need = sorted(set(DEV[['clip1', 'clip2', 'clip3']].values.ravel()))
miss = [c for c in need if c not in G8]
assert not miss, f"{len(miss)} clips missing from G8a, e.g. {miss[:3]}"


def softmax(z):
    e = np.exp(z - z.max(-1, keepdims=True))
    return e / e.sum(-1, keepdims=True)


# per-clip frame features except the PCA part (18 dims), and the raw HSEmotion embeddings
FB, EMB = {}, {}
for c in need:
    fs = G8[c]['faces']
    if not fs:
        FB[c], EMB[c] = np.zeros((0, 18), np.float32), None
        continue
    dur = max(G8[c]['meta'].get('duration') or 0.0, 1e-3)
    fer = np.stack([d['fer'] for d in fs]).astype(np.float32)
    box = np.stack([d['box'] for d in fs])
    FB[c] = np.concatenate([
        softmax(fer[:, :8]), fer[:, 8:10], np.stack([d['pose'] for d in fs]) / 90.0,
        np.stack([(box[:, 0] + box[:, 2]) / 2, (box[:, 1] + box[:, 3]) / 2,
                  np.sqrt(np.clip((box[:, 2] - box[:, 0]) * (box[:, 3] - box[:, 1]), 0, None))], 1),
        np.nan_to_num(np.array([[d['mouth'] * 10] for d in fs], np.float32)),
        np.array([[min(d['t'] / dur, 1.0)] for d in fs], np.float32)], 1).astype(np.float32)
    EMB[c] = np.stack([d['fer_emb'] for d in fs]) if all('fer_emb' in d for d in fs) else None
HAS_EMB = all(EMB[c] is not None for c in need if len(G8[c]['faces']))
FDIM = (PCA_DIM if HAS_EMB else 0) + 18
print(f"HSEmotion embedding available: {HAS_EMB} | frame feature dim {FDIM}")


def pick_frames(idx, cap):
    return idx if len(idx) <= cap else [idx[i] for i in np.linspace(0, len(idx) - 1, cap).astype(int)]


def sync(c, face_ids):
    # correlation of mouth opening with the audio energy envelope, and mouth variability, for a set of faces
    A = G8[c]['audio']
    fs = [G8[c]['faces'][j] for j in face_ids]
    fs = [d for d in fs if np.isfinite(d['mouth'])]
    if A is None or len(fs) < 4:
        return 0.0, 0.0
    m = np.array([d['mouth'] for d in fs])
    env = A['env']
    e = np.array([env[min(int(d['t'] * 10), len(env) - 1)] for d in fs]) if len(env) else np.zeros(len(fs))
    r = float(np.corrcoef(m, e)[0, 1]) if m.std() > 1e-6 and e.std() > 1e-9 else 0.0
    return r, float(m.std() * 10)


def mean12(c_faces, n_sampled):
    if not c_faces:
        return np.zeros(12, np.float32)
    fer = np.stack([d['fer'] for d in c_faces]).astype(np.float32)
    v = np.concatenate([softmax(fer[:, :8]), fer[:, 8:10]], 1).mean(0)
    return np.concatenate([v, [1.0, len({d['frame'] for d in c_faces}) / max(n_sampled, 1)]]).astype(np.float32)


NVOICE = 3 * 2 + 3
SLOT, PSLOT = {}, {}                      # (n, role, clip) / (n, clip) -> (clip id, face indices)
FMASK = np.zeros((N, 3, 3, MAXF), bool); PMASK = np.zeros((N, 1, 3, MAXF_POOL), bool)
VOI = np.zeros((N, 3, NVOICE), np.float32)
LRF = np.zeros((N, 60), np.float32)       # G6b-style role means (late fusion and the joint-branch arm)
CENT_COS = np.full(N, np.nan, np.float32)
for n, row in enumerate(tqdm(DEV.itertuples(), total=N, desc='roles')):
    cl = [row.clip1, row.clip2, row.clip3]
    items = [(k, j) for k, c in enumerate(cl) for j in range(len(G8[c]['faces']))]
    lab = np.zeros(len(items), int)
    E = np.stack([G8[cl[k]]['faces'][j]['arc'] for k, j in items]).astype(np.float32) if items else None
    if len(items) > 1:
        lab = AgglomerativeClustering(n_clusters=None, metric='cosine', linkage='average',
                                      distance_threshold=1 - SAME_PERSON_COS).fit_predict(E)
    frames = defaultdict(set)
    for (k, j), p in zip(items, lab):
        frames[(k, p)].add(G8[cl[k]]['faces'][j]['frame'])
    ids3 = sorted({p for (k, p) in frames if k == 2}, key=lambda p: -len(frames[(2, p)]))
    A = ids3[0] if ids3 else None
    L = ids3[1] if len(ids3) > 1 else None
    if A is not None and L is not None:
        ca, cb = E[lab == A].mean(0), E[lab == L].mean(0)
        CENT_COS[n] = float(ca @ cb / (np.linalg.norm(ca) * np.linalg.norm(cb) + 1e-9))
    role = lambda p: 0 if p == A else (1 if p == L else 2)
    by = defaultdict(list)
    for (k, j), p in zip(items, lab):
        by[(role(p), k)].append(j)
        by[('pool', k)].append(j)
    for k, c in enumerate(cl):
        for r in range(3):
            idx = pick_frames(sorted(by[(r, k)], key=lambda j: G8[c]['faces'][j]['t']), MAXF)
            SLOT[(n, r, k)] = (c, idx); FMASK[n, r, k, :len(idx)] = True
        idx = pick_frames(sorted(by[('pool', k)], key=lambda j: G8[c]['faces'][j]['t']), MAXF_POOL)
        PSLOT[(n, k)] = (c, idx); PMASK[n, 0, k, :len(idx)] = True
        v = [x for r in range(3) for x in sync(c, by[(r, k)])]
        a3, ak = G8[cl[2]]['audio'], G8[c]['audio']
        ok = a3 is not None and ak is not None and a3.get('ecapa') is not None and ak.get('ecapa') is not None
        vcos = float(a3['ecapa'].astype(np.float32) @ ak['ecapa'].astype(np.float32)) if ok else 0.0
        VOI[n, k] = v + [vcos, float(ok), len(set(lab)) / 5.0]

    def faces_of(k, p):
        return [G8[cl[k]]['faces'][j] for (kk, j), q in zip(items, lab) if kk == k and q == p]

    def dominant(k):
        ns = G8[cl[k]]['meta']['n_sampled']
        cand = sorted({q for (kk, q) in frames if kk == k}, key=lambda q: -len(frames[(k, q)]))
        return cand[0] if cand and len(frames[(k, cand[0])]) / max(ns, 1) >= DOMINANT_MIN_FRAC else None

    n3 = G8[cl[2]]['meta']['n_sampled']
    blocks = [mean12(faces_of(2, A) if A is not None else [], n3), mean12(faces_of(2, L) if L is not None else [], n3),
              mean12((faces_of(0, L) + faces_of(1, L)) if L is not None else [], n3)]
    for k in (1, 0):
        d = dominant(k)
        blocks.append(mean12(faces_of(k, d) if d is not None else [], G8[cl[k]]['meta']['n_sampled']))
    LRF[n] = np.concatenate(blocks)

VIS = FMASK[:, 1, 2].any(-1)
print(f"A found in III {FMASK[:, 0, 2].any(-1).mean() * 100:.1f}% | listener visible in III {VIS.mean() * 100:.1f}% | "
      f"listener also in I/II {FMASK[:, 1, :2].any((-1, -2)).mean() * 100:.1f}%")


def build_face_tensors(fit_clips):
    # PCA of the HSEmotion embedding fitted on faces of the training clips of the current fold only
    pca, var = None, None
    if HAS_EMB:
        pool = np.concatenate([EMB[c] for c in fit_clips if EMB.get(c) is not None])
        pick = np.random.default_rng(0).choice(len(pool), min(60000, len(pool)), replace=False)
        pca = PCA(PCA_DIM, random_state=0).fit(pool[pick].astype(np.float32))
        var = float(pca.explained_variance_ratio_.sum())
    FV = {}
    for c in need:
        if len(FB[c]) == 0:
            FV[c] = np.zeros((0, FDIM), np.float32)
        else:
            FV[c] = np.concatenate([pca.transform(EMB[c].astype(np.float32)), FB[c]], 1) if HAS_EMB else FB[c]
    Fa = np.zeros((N, 3, 3, MAXF, FDIM), np.float16)
    Pa = np.zeros((N, 1, 3, MAXF_POOL, FDIM), np.float16)
    for (n, r, k), (c, idx) in SLOT.items():
        if idx:
            Fa[n, r, k, :len(idx)] = FV[c][idx]
    for (n, k), (c, idx) in PSLOT.items():
        if idx:
            Pa[n, 0, k, :len(idx)] = FV[c][idx]
    return torch.tensor(Fa, device=DEVICE), torch.tensor(Pa, device=DEVICE), var

## Models

In [ ]:
T = lambda a, dt=None: torch.tensor(a, device=DEVICE) if dt is None else torch.tensor(a, dtype=dt, device=DEVICE)
FMASK, PMASK, VOI = T(FMASK), T(PMASK), T(VOI)
FACE = POOL = LRFZ = None             # set per fold
CLIPIDX = T([[CIDX[c] for c in r] for r in DEV[['clip1', 'clip2', 'clip3']].values])
TXT, AUD, AFD = FEAT['text'][CLIPIDX], F.normalize(FEAT['audio'][CLIPIDX], dim=-1), FEAT['afound'][CLIPIDX].float()
fm = FEAT['fmask'][CLIPIDX].unsqueeze(-1).float()
SCN = torch.cat([FEAT['ori'][CLIPIDX].mean(2), (FEAT['face'][CLIPIDX] * fm).sum(2) / fm.sum(2).clamp(min=1)], -1)
ZREC = torch.zeros(N, 3, N_REC, device=DEVICE)
YB, YA = T(DEV.yB.values), T(DEV.yA.values)


class FramePool(nn.Module):
    def __init__(self, fin, d):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(fin), nn.Linear(fin, d), nn.GELU(), nn.Linear(d, d))
        self.score = nn.Linear(d, 1)

    def forward(self, x, m):                      # x [..., F, fin], m [..., F]
        h = self.proj(x.float())
        a = self.score(h).squeeze(-1).masked_fill(~m, -1e4)
        w = torch.softmax(a, -1) * m.float()
        return (w.unsqueeze(-1) * h).sum(-2), m.any(-1)


class RoleNet(nn.Module):
    def __init__(self, cfg, d=RN['D']):
        super().__init__()
        self.cfg, self.R = cfg, (3 if cfg['role'] else 1)
        if cfg['faces']:
            self.pool = FramePool(FDIM, d)
            self.absent = nn.Parameter(torch.randn(self.R, 3, d) * 0.02)
            self.face_role = nn.Parameter(torch.randn(self.R, d) * 0.02)
        if cfg['ctx']:
            self.text = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, d))
            self.audio = nn.Sequential(nn.LayerNorm(527), nn.Linear(527, d))
            self.voice = nn.Linear(NVOICE, d)
            self.scene = nn.Sequential(nn.LayerNorm(1024), nn.Linear(1024, d))
            self.ctx_role = nn.Parameter(torch.randn(2, d) * 0.02)
        self.clip_emb = nn.Parameter(torch.randn(3, d) * 0.02)
        self.query = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, RN['heads'], 4 * d, RN['dropout'], batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, RN['layers'], enable_nested_tensor=False)
        mk = lambda: nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, 7))
        self.head = mk()
        self.head_face = mk() if cfg['aux'] and cfg['faces'] else None
        self.head_ctx = mk() if cfg['aux'] and cfg['ctx'] else None
        self.head_A = mk() if cfg['aux'] and cfg['faces'] and cfg['role'] else None

    def forward(self, ix, train=False):
        B, groups, aux = len(ix), [], {}
        if self.cfg['faces']:
            x, m = (FACE[ix], FMASK[ix]) if self.R == 3 else (POOL[ix], PMASK[ix])
            h, present = self.pool(x, m)                                      # [B, R, 3, d]
            h = torch.where(present.unsqueeze(-1), h, self.absent.unsqueeze(0).expand(B, -1, -1, -1))
            h = h + self.face_role[None, :, None] + self.clip_emb[None, None]
            ft = h.reshape(B, self.R * 3, -1)
            groups.append(ft)
            if self.head_face is not None:
                aux['face'] = (self.head_face(ft.mean(1)), YB[ix], RN['aux_w'])
            if self.head_A is not None:
                tA = torch.where(present[:, 0, 2], YA[ix], torch.full_like(YA[ix], -100))
                aux['A'] = (self.head_A(h[:, 0, 2]), tA, RN['a_w'])
        if self.cfg['ctx']:
            spk_ = self.text(TXT[ix]) + self.audio(AUD[ix]) * AFD[ix].unsqueeze(-1) + self.voice(VOI[ix]) + self.ctx_role[0]
            scn = self.scene(SCN[ix]) + self.ctx_role[1]
            ct = torch.cat([spk_ + self.clip_emb, scn + self.clip_emb], 1)     # [B, 6, d]
            groups.append(ct)
            if self.head_ctx is not None:
                aux['ctx'] = (self.head_ctx(ct.mean(1)), YB[ix], RN['aux_w'])
        toks = torch.cat([self.query.expand(B, -1, -1)] + groups, 1)
        valid = torch.ones(toks.shape[:2], dtype=torch.bool, device=toks.device)
        if train and self.cfg['mdrop'] and len(groups) == 2:
            u = torch.rand(B, device=toks.device)
            drop_ctx = u < RN['p_drop_ctx']
            drop_face = (u >= RN['p_drop_ctx']) & (u < RN['p_drop_ctx'] + RN['p_drop_face'])
            nf = groups[0].shape[1]
            valid[:, 1:1 + nf] &= ~drop_face.unsqueeze(1)
            valid[:, 1 + nf:] &= ~drop_ctx.unsqueeze(1)
        out = self.enc(toks, src_key_padding_mask=~valid)
        return self.head(out[:, 0]), aux


class B1Wrap(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = Forecaster(use_raw=True, use_traj=False)

    def forward(self, ix, train=False):
        return self.f(CLIPIDX[ix], ZREC[ix]), {}


class B1FaceWrap(nn.Module):
    # B1 + the 60-d G6b-style role-face vector through a zero-initialised branch, trained jointly (the G7b design)
    def __init__(self, n_face=60, d=512):
        super().__init__()
        self.f = Forecaster(use_raw=True, use_traj=False, d=d)
        self.face = nn.Sequential(nn.Linear(n_face, d // 2), nn.GELU(), nn.Dropout(0.3), nn.Linear(d // 2, d))
        nn.init.zeros_(self.face[-1].weight); nn.init.zeros_(self.face[-1].bias)

    def forward(self, ix, train=False):
        b, clip_idx = self.f, CLIPIDX[ix]
        B, n = clip_idx.shape
        feats = gather(clip_idx)
        tok = b.enc({k: v.reshape(B * n, *v.shape[2:]) for k, v in feats.items()}).reshape(B, n, -1)
        h = b.inter(tok + b.clip_pos[:, 3 - n:])
        return b.head(h.mean(1) + self.face(LRFZ[ix])), {}


HP = {'b1': dict(lr=LR, wd=WEIGHT_DECAY, epochs=FC_EPOCHS, patience=PATIENCE, batch=FC_BATCH),
      'role': dict(lr=RN['lr'], wd=RN['wd'], epochs=RN['epochs'], patience=RN['patience'], batch=RN['batch'])}
HP['b1face'] = HP['b1']
MAKE = {'b1': lambda cfg: B1Wrap(), 'b1face': lambda cfg: B1FaceWrap(), 'role': lambda cfg: RoleNet(cfg)}
print("parameters:", {n: f"{sum(p.numel() for p in MAKE[k](c).parameters()) / 1e6:.2f}M" for n, k, c in EXPERIMENTS})


def predict(model, ix, bs=256):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(ix), bs):
            out.append(F.softmax(model(ix[i:i + bs])[0], -1).cpu())
    return torch.cat(out).numpy()


def train_eval(kind, cfg, tr, dev, te, seed):
    seed_all(seed)
    hp = HP[kind]
    model = MAKE[kind](cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=hp['lr'], weight_decay=hp['wd'])
    y_dev = YB[dev].cpu().numpy()
    best, best_state, bad = -1, None, 0
    for ep in range(hp['epochs']):
        model.train()
        perm = tr[torch.randperm(len(tr), device=DEVICE)]
        for i in range(0, len(perm), hp['batch']):
            j = perm[i:i + hp['batch']]
            logits, aux = model(j, train=True)
            loss = F.cross_entropy(logits, YB[j])
            for l, t, w in aux.values():
                if (t >= 0).any():
                    loss = loss + w * F.cross_entropy(l, t, ignore_index=-100)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        u = war_uar(predict(model, dev).argmax(1), y_dev, 7)[1]
        if u > best:
            best, bad = u, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= hp['patience']:
                break
    model.load_state_dict(best_state)
    return predict(model, te), best

## Feature blocks and folds

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

sizes = DEV.source_folder.value_counts()
order = list(sizes.index)
random.Random(0).shuffle(order)
order = sorted(order, key=lambda e: -sizes[e])
load_, FOLD = [0] * N_OUTER, {}
for e in order:
    f = int(np.argmin(load_)); FOLD[e] = f; load_[f] += sizes[e]
fold_of_row = DEV.source_folder.map(FOLD).values
y_all, yA_all = DEV.yB.values.astype(int), DEV.yA.values.astype(int)
src = DEV.source_folder.values
print("fold sizes (MCIS):", load_, "| MCIS with Y_B = Y_A:", f"{np.mean(y_all == yA_all) * 100:.1f}%",
      "| rows with an A label:", int((yA_all >= 0).sum()))

BLK = {'A3': LRF[:, 0:12], 'L3': LRF[:, 12:24], 'L12': LRF[:, 24:36], 'D2': LRF[:, 36:48], 'D1': LRF[:, 48:60]}
BLK['POOL'] = np.mean([BLK[k] for k in ('A3', 'L3', 'L12', 'D2', 'D1')], 0)        # noRole: one pooled face block
TXT3 = TXT[:, 2].float().cpu().numpy()
AUD3 = AUD[:, 2].float().cpu().numpy()
print("feature blocks:", {k: v.shape for k, v in BLK.items()}, "| text III", TXT3.shape, "| audio III", AUD3.shape)

## Model fitting

In [ ]:
def tt(x):
    return torch.as_tensor(np.asarray(x, np.float64))


def nll_rows(logp, y):
    return -logp[np.arange(len(y)), y]


def edge(v, grid):
    return v in (grid[0], grid[-1])


def fit_lr_proba(Xtr, ytr, gtr, Xte):
    def proba(m, X):
        p = np.full((len(X), 7), 1e-6); p[:, m.classes_] = m.predict_proba(X)
        return p / p.sum(1, keepdims=True)
    gkf = list(GroupKFold(5).split(Xtr, ytr, gtr))
    score = {}
    for C in C_GRID:
        score[C] = np.mean([nll_rows(np.log(proba(LogisticRegression(C=C, max_iter=3000).fit(Xtr[a], ytr[a]), Xtr[b])),
                                     ytr[b]).mean() for a, b in gkf])
    C = min(score, key=score.get)
    p_tr = np.zeros((len(Xtr), 7))
    for a, b in gkf:
        p_tr[b] = proba(LogisticRegression(C=C, max_iter=3000).fit(Xtr[a], ytr[a]), Xtr[b])
    return p_tr, proba(LogisticRegression(C=C, max_iter=3000).fit(Xtr, ytr), Xte), C


def head_logp(kind, P, Xe, Xg, LPm):
    # kind 'mix': g p_mirror + (1 - g) softmax(Xe W + b), g = sigmoid(Xg v + c); 'add': softmax(a LPm + Xe W + b)
    if kind == 'add':
        return F.log_softmax(P['a'] * LPm + Xe @ P['W'] + P['b'], -1)
    s = Xg @ P['v'] + P['c']
    lo = F.log_softmax(Xe @ P['W'] + P['b'], -1)
    return torch.logsumexp(torch.stack([F.logsigmoid(s).unsqueeze(-1) + LPm, F.logsigmoid(-s).unsqueeze(-1) + lo]), 0)


def fit_head(kind, Xe, Xg, LPm, y, lam_w, lam_v, c0):
    torch.manual_seed(0)
    Xe, Xg, LPm, yt = tt(Xe), tt(Xg), tt(LPm), torch.as_tensor(np.asarray(y), dtype=torch.long)
    z_ = lambda *sh: torch.zeros(*sh, dtype=torch.float64, requires_grad=True)
    P = {'W': z_(Xe.shape[1], 7), 'b': z_(7)}
    if kind == 'mix':
        P.update(v=z_(Xg.shape[1]), c=torch.full((1,), float(c0), dtype=torch.float64, requires_grad=True))
    else:
        P.update(a=torch.ones(1, dtype=torch.float64, requires_grad=True))
    opt = torch.optim.LBFGS(list(P.values()), lr=1, max_iter=500, line_search_fn='strong_wolfe')

    def closure():
        opt.zero_grad()
        loss = F.nll_loss(head_logp(kind, P, Xe, Xg, LPm), yt) + lam_w * (P['W'] ** 2).sum()
        if kind == 'mix':
            loss = loss + lam_v * (P['v'] ** 2).sum()
        loss.backward()
        return loss
    opt.step(closure)
    return {k: v.detach() for k, v in P.items()}


def logp_of(kind, P, Xe, Xg, LPm):
    return head_logp(kind, P, tt(Xe), tt(Xg), tt(LPm)).numpy()


def run_fold(tr, te, diag):
    zs = lambda M: (M - M[tr].mean(0)) / (M[tr].std(0) + 1e-6)
    pt = PCA(PCA_TXT, random_state=0).fit(TXT3[tr]); pa = PCA(PCA_AUD, random_state=0).fit(AUD3[tr])
    T3, A3a = zs(pt.transform(TXT3)), zs(pa.transform(AUD3))
    B = {k: zs(v) for k, v in BLK.items()}
    gkf = list(GroupKFold(5).split(tr, y_all[tr], src[tr]))
    X_own = np.concatenate([B['L3'], B['L12']], 1)
    X_all = np.concatenate([B['A3'], B['L3'], B['L12'], B['D2'], B['D1'], T3, A3a], 1)
    out = {}
    if diag == 'oracle':
        onehot = np.eye(7)[yA_all]

        def LP(eps):
            return np.log((1 - eps) * onehot + eps / 7)
        sc = {}
        for eps in EPS_GRID:
            Lp = LP(eps)
            for lw in LAM_W:
                sc[(eps, lw)] = np.mean([nll_rows(logp_of('add', fit_head('add', X_all[tr[a]], None, Lp[tr[a]], y_all[tr[a]],
                                                                                 lw, 0, 0), X_all[tr[b]], X_all[tr[b]], Lp[tr[b]]),
                                                  y_all[tr[b]]).mean() for a, b in gkf])
        eps_add, lw_add = min(sc, key=sc.get)
        out.update(eps_add=eps_add, eps_add_edge=edge(eps_add, EPS_GRID))
        EPS_MIX, LPm = EPS_GRID, None
    else:
        Xm = np.concatenate([B['A3'], T3, A3a], 1)
        p_tr, p_all, Cm = fit_lr_proba(Xm[tr], yA_all[tr], src[tr], Xm)
        pm = p_all.copy(); pm[tr] = p_tr
        LPm = np.log(np.clip(pm, 1e-6, None))
        out.update(C_mirror=Cm, C_edge=edge(Cm, C_GRID),
                   p_mirror_UAR_on_A=war_uar(pm[te].argmax(1), yA_all[te], 7)[1])
        sc = {lw: np.mean([nll_rows(logp_of('add', fit_head('add', X_all[tr[a]], None, LPm[tr[a]], y_all[tr[a]], lw, 0, 0),
                                            X_all[tr[b]], X_all[tr[b]], LPm[tr[b]]), y_all[tr[b]]).mean() for a, b in gkf])
              for lw in LAM_W}
        lw_add = min(sc, key=sc.get)
        EPS_MIX = [None]
    LPm_add = LP(eps_add) if diag == 'oracle' else LPm
    sc = {}
    for eps in EPS_MIX:
        Lp = LP(eps) if eps is not None else LPm
        Xg_ = np.concatenate([X_all, Lp], 1)
        for lw in LAM_W:
            for lv in LAM_V:
                s = []
                for a, b in gkf:
                    share = np.clip(np.mean(y_all[tr[a]] == yA_all[tr[a]]), 0.01, 0.99)
                    P = fit_head('mix', X_own[tr[a]], Xg_[tr[a]], Lp[tr[a]], y_all[tr[a]], lw, lv, np.log(share / (1 - share)))
                    s.append(nll_rows(logp_of('mix', P, X_own[tr[b]], Xg_[tr[b]], Lp[tr[b]]), y_all[tr[b]]).mean())
                sc[(eps, lw, lv)] = np.mean(s)
    eps_mix, lw_mix, lv_mix = min(sc, key=sc.get)
    if diag == 'oracle':
        LPm = LP(eps_mix)
        out.update(eps_mix=eps_mix, eps_mix_edge=edge(eps_mix, EPS_GRID))
    Xg = np.concatenate([X_all, LPm], 1)
    share = np.clip(np.mean(y_all[tr] == yA_all[tr]), 0.01, 0.99)
    Pm = fit_head('mix', X_own[tr], Xg[tr], LPm[tr], y_all[tr], lw_mix, lv_mix, np.log(share / (1 - share)))
    Pa = fit_head('add', X_all[tr], None, LPm_add[tr], y_all[tr], lw_add, 0, 0)
    out['mix'] = logp_of('mix', Pm, X_own[te], Xg[te], LPm[te])
    out['add'] = logp_of('add', Pa, X_all[te], X_all[te], LPm_add[te])
    out['gate'] = torch.sigmoid(tt(Xg[te]) @ Pm['v'] + Pm['c']).numpy()
    out.update(lam_w_mix=lw_mix, lam_v_mix=lv_mix, lam_w_add=lw_add, alpha=float(Pa['a']),
               edges=f"Wmix {edge(lw_mix, LAM_W)} | vmix {edge(lv_mix, LAM_V)} | Wadd {edge(lw_add, LAM_W)}")
    return out

## 5-fold episode cross-validation

In [ ]:
DIAGS = ['oracle', 'AvsB']
OOF = {d: {k: np.zeros((N, 7)) for k in ('mix', 'add')} for d in DIAGS}
GATE = {d: np.zeros(N) for d in DIAGS}
info = []
t0 = time.time()
for f in range(N_OUTER):
    tr, te = np.where(fold_of_row != f)[0], np.where(fold_of_row == f)[0]
    for d in DIAGS:
        o = run_fold(tr, te, d)
        OOF[d]['mix'][te], OOF[d]['add'][te], GATE[d][te] = o['mix'], o['add'], o['gate']
        rec = {k: v for k, v in o.items() if k not in ('mix', 'add', 'gate')}
        rec.update(fold=f, diag=d, gate_mean=o['gate'].mean(), gate_sd=o['gate'].std())
        info.append(rec)
        print(f"fold {f} {d:<6}: " + " | ".join(f"{k} {v}" for k, v in rec.items() if k not in ('fold', 'diag'))
              + f" | {(time.time() - t0) / 60:.1f} min", flush=True)
INFO = pd.DataFrame(info)
INFO.to_csv(f"{OUT_DIR}/g32_fold_log.csv", index=False)
np.savez(f"{OUT_DIR}/g32_oof.npz", sample_id=DEV.sample_id.values, fold=fold_of_row, y=y_all, yA=yA_all, src=src,
         **{f"{d}_{k}": a for d, dd in OOF.items() for k, a in dd.items()}, **{f"{d}_gate": g for d, g in GATE.items()})
print("saved g32_oof.npz and g32_fold_log.csv")

## Results and the fixed reading rule

In [ ]:
MIRROR = (y_all == yA_all)
L = {d: {k: nll_rows(a, y_all) for k, a in dd.items()} for d, dd in OOF.items()}
gidx = [np.where(src == e)[0] for e in np.unique(src)]
rng = np.random.default_rng(0)
DR = {d: [] for d in DIAGS}
AU = {d: [] for d in DIAGS}
for _ in range(N_BOOT):
    idx = np.concatenate([gidx[i] for i in rng.integers(0, len(gidx), len(gidx))])
    for d in DIAGS:
        DR[d].append(L[d]['add'][idx].mean() - L[d]['mix'][idx].mean())
        AU[d].append(roc_auc_score(MIRROR[idx], GATE[d][idx]))
CI = lambda x: np.percentile(x, [2.5, 97.5])
PASS = {}
for d in DIAGS:
    dn, (lo, hi) = L[d]['add'].mean() - L[d]['mix'].mean(), CI(DR[d])
    au, (alo, ahi) = roc_auc_score(MIRROR, GATE[d]), CI(AU[d])
    PASS[d] = lo > 0 and alo > 0.5
    q = np.percentile(GATE[d], [5, 25, 50, 75, 95])
    print(f"\n== ({'i' if d == 'oracle' else 'ii'}) {d} ==")
    print(f"  NLL mixture {L[d]['mix'].mean():.4f} | additive {L[d]['add'].mean():.4f} | "
          f"UAR mixture {war_uar(OOF[d]['mix'].argmax(1), y_all, 7)[1]:.2f} | additive {war_uar(OOF[d]['add'].argmax(1), y_all, 7)[1]:.2f}")
    print(f"  (a) ΔNLL = NLL(additive) − NLL(mixture) {dn:+.4f} [{lo:+.4f}, {hi:+.4f}]")
    print(f"  (b) AUC(g → Y_B = Y_A) {au:.3f} [{alo:.3f}, {ahi:.3f}]")
    print(f"  gate quantiles 5/25/50/75/95%: {np.round(q, 3)} | mean on mirror {GATE[d][MIRROR].mean():.3f} vs shift "
          f"{GATE[d][~MIRROR].mean():.3f}")
    print(f"  grid edges per fold: {INFO[INFO.diag == d].edges.tolist()}")
    print(f"  -> {'PASSED' if PASS[d] else 'NOT PASSED'}")
if not PASS['oracle']:
    verdict = "even with A's true emotion the gate cannot tell when B mirrors → STOP direction A"
elif not PASS['AvsB']:
    verdict = "the structure has value; the bottleneck is recognising A's emotion → contribution 2 must improve the A branch"
else:
    verdict = "both passed → write the 2×2 notebook"
print(f"\n== G32 joint reading (fixed rule; second look after G31): {verdict} ==")
pd.DataFrame([{'diag': d, 'model': k, 'NLL': L[d][k].mean(), 'UAR': war_uar(OOF[d][k].argmax(1), y_all, 7)[1]}
              for d in DIAGS for k in L[d]]).to_csv(f"{OUT_DIR}/g32_summary.csv", index=False)
print("saved g32_summary.csv")